# SOLiD-PINN v0.9.0b — corrected mesh bridge
CPU, Run all. No Drive mount or GitHub token. Eight tests, five mesh levels, 216 fits if mesh gate passes. Synthetic reduced model only. Download JSON before ending runtime.


In [ ]:
from google.colab import files
from pathlib import Path
import subprocess,sys,os,tempfile,json,hashlib
REV="1c6726d3d22fe945f961ce65a4f6f8aa62776f5e"
repo=Path(tempfile.mkdtemp(prefix="solid-v090b-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==REV
hashes={"mesh_bridge.py":"b6ad34bb01079e34e1134daf801a48f36f4695c3b5f57f86aa6af3e68a2fbf77","test_mesh_bridge.py":"4a80bcedd026888d8feff2ad602fdbaa13d5ba1307260582f20236e1bdc1cd86","hybrid_p2d.py":"20aaeede052a29a2b65b1cd322b9fade3b4094fd4e1564d15787d7e240c51f18","test_hybrid_p2d.py":"08f898a7771aea81ebb41d92d375fe758b1dc8d7663a45321096062d5be4295f"}
for name,digest in hashes.items(): assert hashlib.sha256((repo/"scripts"/name).read_bytes()).hexdigest()==digest
env=dict(os.environ,OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_mesh_bridge.py","scripts/test_hybrid_p2d.py"],cwd=repo,env=env,check=True)
out=repo/"results.json"
subprocess.run([sys.executable,"scripts/mesh_bridge.py","--out",str(out)],cwd=repo,env=env,check=True)
d=json.loads(out.read_text())
assert set(d["mesh"])=={"train","test","extra"}
for v in d["mesh"].values():
    assert v["time_reference_mv"]<1e-5 and len(v["checks"])==5
    for c in v["checks"].values():
        assert c["mass"]<1e-9 and c["charge"]<1e-9 and 0<c["min_c"]<=c["max_c"]<1 and 0<c["min_surface"]<=c["max_surface"]<1
gate=all(max(list(v["delta_mv"].values())[-2:])<=1 for v in d["mesh"].values())
assert d["mesh_gate_passed"]==gate
expected={(t,n,s,mesh,m) for t in ("null","rest","peak") for n in (1,5) for s in (11,23,37) for mesh in ((8,12),(32,48),(128,192)) for m in ("calibrated","linear","rbf","oracle")}
if gate:
    assert len(d["metrics"])==216 and all(x["success"] for x in d["metrics"])
    assert {(x["truth"],x["noise"],x["seed"],tuple(x["mesh"]),x["method"]) for x in d["metrics"]}==expected
else: assert not d["metrics"]
d.update(source_revision=REV,source_hashes=hashes,targeted_tests_passed=8)
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
root=Path("/content/SOLiD-PINN/v090b/handoff");root.mkdir(parents=True,exist_ok=True)
dest=root/f"SOLiD-PINN_v090b_colab_{sha[:20]}.json"
if dest.exists(): assert dest.read_bytes()==raw
else:
    tmp=dest.with_suffix(".part");tmp.write_bytes(raw);tmp.replace(dest)
assert dest.read_bytes()==raw
print("MESH GATE",gate,"LOCAL FILE VERIFIED",dest.name,sha)
try: files.download(str(dest))
except Exception as e: print("Download manually from Files panel:",str(dest),str(e))
